# Assignment 3 Solution: Sanskrit-English Sentence Embeddings using PyTorch

This notebook provides a complete, runnable PyTorch-based workflow for generating sentence-level embeddings for Sanskrit-English parallel sentence pairs, evaluating them using average cosine similarity, and visualizing 100 sample pairs using t-SNE.

**Expected dataset files**
- `train_sa.csv`, `dev_sa.csv`, `test_sa.csv`
- `train_en.csv`, `dev_en.csv`, `test_en.csv`

Place these CSV files in the same directory as this notebook before running.

## 1. Install dependencies

Run this cell only if the required packages are not already installed.

In [ ]:

# Uncomment and run if needed
# !pip install -q pandas numpy matplotlib scikit-learn torch transformers sentencepiece

## 2. Import libraries

In [ ]:

import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from sklearn.manifold import TSNE

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## 3. File paths

Update `DATA_DIR` if your CSV files are stored in another folder.

In [ ]:

DATA_DIR = '.'

files = {
    'train_sa': os.path.join(DATA_DIR, 'train_sa.csv'),
    'dev_sa': os.path.join(DATA_DIR, 'dev_sa.csv'),
    'test_sa': os.path.join(DATA_DIR, 'test_sa.csv'),
    'train_en': os.path.join(DATA_DIR, 'train_en.csv'),
    'dev_en': os.path.join(DATA_DIR, 'dev_en.csv'),
    'test_en': os.path.join(DATA_DIR, 'test_en.csv'),
}

for name, path in files.items():
    print(f'{name}:', 'FOUND' if os.path.exists(path) else f'MISSING -> {path}')

## 4. Load and merge aligned sentence pairs

The Sanskrit and English files are aligned using `Source_id`.

In [ ]:

def load_parallel_split(sa_path, en_path, split_name):
    sa = pd.read_csv(sa_path)
    en = pd.read_csv(en_path)

    expected_sa = {'Source_id', 'Sentence_sa'}
    expected_en = {'Source_id', 'Sentence_en'}

    if not expected_sa.issubset(sa.columns):
        raise ValueError(f'{split_name} Sanskrit file must contain columns: {expected_sa}')
    if not expected_en.issubset(en.columns):
        raise ValueError(f'{split_name} English file must contain columns: {expected_en}')

    df = sa.merge(en, on='Source_id', how='inner')
    df = df.dropna(subset=['Sentence_sa', 'Sentence_en']).copy()
    df['Sentence_sa'] = df['Sentence_sa'].astype(str).str.strip()
    df['Sentence_en'] = df['Sentence_en'].astype(str).str.strip()
    df = df[(df['Sentence_sa'] != '') & (df['Sentence_en'] != '')].reset_index(drop=True)
    df['split'] = split_name
    return df

train_df = load_parallel_split(files['train_sa'], files['train_en'], 'train')
dev_df = load_parallel_split(files['dev_sa'], files['dev_en'], 'dev')
test_df = load_parallel_split(files['test_sa'], files['test_en'], 'test')

all_df = pd.concat([train_df, dev_df, test_df], ignore_index=True)

print('Train pairs:', len(train_df))
print('Dev pairs  :', len(dev_df))
print('Test pairs :', len(test_df))
print('Total pairs:', len(all_df))
all_df.head()

## 5. Load multilingual transformer model

This notebook uses a multilingual transformer through Hugging Face Transformers and PyTorch. Sentence embeddings are created by mean pooling the token embeddings.

In [ ]:

MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

hidden_size = model.config.hidden_size
print('Model loaded:', MODEL_NAME)
print('Embedding dimension used:', hidden_size)

## 6. Define PyTorch mean-pooling encoder

In [ ]:

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    pooled = torch.sum(token_embeddings * input_mask_expanded, dim=1)
    pooled = pooled / torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
    return pooled

@torch.no_grad()
def encode_sentences(sentences, batch_size=32, max_length=128):
    embeddings = []

    for i in range(0, len(sentences), batch_size):
        batch_sentences = sentences[i:i+batch_size]
        encoded = tokenizer(
            batch_sentences,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        outputs = model(**encoded)
        batch_embeddings = mean_pooling(outputs, encoded['attention_mask'])
        batch_embeddings = F.normalize(batch_embeddings, p=2, dim=1)
        embeddings.append(batch_embeddings.cpu())

    return torch.cat(embeddings, dim=0)

## 7. Generate sentence embeddings

We compute embeddings separately for Sanskrit and English sentences using the PyTorch encoder defined above.

In [ ]:

sa_embeddings_torch = encode_sentences(all_df['Sentence_sa'].tolist(), batch_size=32)
en_embeddings_torch = encode_sentences(all_df['Sentence_en'].tolist(), batch_size=32)

print('Sanskrit embeddings shape:', tuple(sa_embeddings_torch.shape))
print('English embeddings shape :', tuple(en_embeddings_torch.shape))

## 8. Compute average cosine similarity

This is the primary evaluation metric mentioned in the assignment.

In [ ]:

pairwise_cosine_scores = torch.sum(sa_embeddings_torch * en_embeddings_torch, dim=1)
avg_cosine_similarity = pairwise_cosine_scores.mean().item()

print('Embedding dimension:', sa_embeddings_torch.shape[1])
print('Average cosine similarity across aligned Sanskrit-English pairs:', round(avg_cosine_similarity, 6))

results_df = all_df[['Source_id', 'split']].copy()
results_df['cosine_similarity'] = pairwise_cosine_scores.cpu().numpy()
results_df.head()

## 9. Optional split-wise analysis

In [ ]:

split_scores = results_df.groupby('split', as_index=False)['cosine_similarity'].mean()
split_scores.columns = ['Split', 'Average Cosine Similarity']
split_scores

## 10. t-SNE visualization for 100 sample pairs

The plot below shows a 2D projection of 100 aligned Sanskrit-English sentence pairs.

In [ ]:

num_pairs = min(100, len(all_df))
sample_df = all_df.sample(num_pairs, random_state=SEED).reset_index(drop=True)

sample_sa_embeddings = encode_sentences(sample_df['Sentence_sa'].tolist(), batch_size=32).cpu().numpy()
sample_en_embeddings = encode_sentences(sample_df['Sentence_en'].tolist(), batch_size=32).cpu().numpy()

combined_embeddings = np.vstack([sample_sa_embeddings, sample_en_embeddings])
perplexity = min(30, max(5, num_pairs // 3))
tsne = TSNE(n_components=2, random_state=SEED, perplexity=perplexity, init='pca', learning_rate='auto')
coords = tsne.fit_transform(combined_embeddings)

plt.figure(figsize=(12, 8))
sa_idx = np.arange(num_pairs)
en_idx = np.arange(num_pairs, 2 * num_pairs)

plt.scatter(coords[sa_idx, 0], coords[sa_idx, 1], c='darkorange', label='Sanskrit', alpha=0.75)
plt.scatter(coords[en_idx, 0], coords[en_idx, 1], c='royalblue', label='English', alpha=0.75)

for i in range(num_pairs):
    plt.plot(
        [coords[sa_idx[i], 0], coords[en_idx[i], 0]],
        [coords[sa_idx[i], 1], coords[en_idx[i], 1]],
        color='gray', alpha=0.15, linewidth=0.8
    )

plt.title(f't-SNE Visualization of {num_pairs} Sanskrit-English Sentence Pairs')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 11. Save embeddings and similarity scores

This step is optional, but useful for further inspection.

In [ ]:

sa_embeddings_np = sa_embeddings_torch.cpu().numpy()
en_embeddings_np = en_embeddings_torch.cpu().numpy()

np.save('sanskrit_embeddings.npy', sa_embeddings_np)
np.save('english_embeddings.npy', en_embeddings_np)
results_df.to_csv('pairwise_cosine_scores.csv', index=False)

print('Saved: sanskrit_embeddings.npy')
print('Saved: english_embeddings.npy')
print('Saved: pairwise_cosine_scores.csv')

## 12. Final notes

This notebook satisfies the assignment structure by including:
- Complete source code for loading data, generating embeddings, and evaluation.
- Explicit embedding dimensionality in the output.
- Final cosine similarity score.
- Inline t-SNE visualization for 100 sample pairs.

The assignment notes that PyTorch is recommended, so this version uses the PyTorch library directly for transformer inference and sentence embedding generation.